In [ ]:
import geemap
import ee

ee.Authenticate(force=True)
ee.Initialize(project='ee-ufmg')

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import geopandas as gpd

gpkg_path = './drive/MyDrive/UTAE/floresta_plantada_ief_2025.gpkg'
shp_path = './drive/MyDrive/UTAE/MG_Municipios_2025/MG_Municipios_2025.shp'

gdf_silv = gpd.read_file(gpkg_path)
gdf_mun = gpd.read_file(shp_path)

# EPSG:31983 - SIRGAS 2000 / UTM zone 23S
gdf_silv = gdf_silv.to_crs(epsg=31983)
gdf_mun = gdf_mun.to_crs(epsg=31983)
gdf_silvi = gpd.read_file(gpkg_path)

In [ ]:
import pandas as pd

gdf_mun['area_total_ha'] = gdf_mun.geometry.area / 10000
silv_mun = gpd.overlay(gdf_silv, gdf_mun, how='intersection')

silv_mun['area_silv_ha'] = silv_mun.geometry.area / 10000

area_por_mun = silv_mun.groupby('NM_MUN')['area_silv_ha'].sum().reset_index()

ranking = pd.merge(gdf_mun[['NM_MUN', 'area_total_ha']], area_por_mun, on='NM_MUN', how='left')

ranking['area_silv_ha'] = ranking['area_silv_ha'].fillna(0)
ranking['proporcao_%'] = (ranking['area_silv_ha'] / ranking['area_total_ha']) * 100
top_10 = ranking.sort_values(by='proporcao_%', ascending=False).head(10)

print("\nCities with largest silviculture area:")
print(top_10[['NM_MUN', 'area_total_ha', 'area_silv_ha', 'proporcao_%']])


Cities with largest silviculture area:
                  NM_MUN  area_total_ha  area_silv_ha  proporcao_%
792           Taiobeiras  122207.609652  33629.537804    27.518366
96    Santana do Paraíso   27581.727858   7167.439254    25.986187
440         Belo Oriente   33523.640033   8408.478559    25.082236
564            Carbonita  145647.350772  35746.627663    24.543274
470  São João do Paraíso  192906.614052  44185.260663    22.905000
356                Ipaba   11336.882135   2453.891408    21.645205
706          Josenópolis   54187.584588  11345.529949    20.937508
452            Veredinha   63209.879866  12602.206430    19.937083
107          Minas Novas  181428.513699  35104.879307    19.349152
213         Itamarandiba  273691.439008  52205.081300    19.074430


In [ ]:
# GEE needs WGS84
top_10_gdf = gdf_mun[['NM_MUN', 'geometry']].merge(top_10, on='NM_MUN', how='inner')
top_10_wgs84 = top_10_gdf.to_crs(epsg=4326)

# GeoDataFrame to Earth Engine FeatureCollection
ee_cities = geemap.geopandas_to_ee(top_10_wgs84)

# Bands used by the paper:
BANDS = ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12']

def mask_s2_clouds(image):
    """Mask clouds using QA60 from Sentinel-2"""
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds.
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    # Keeps pixels where both bits are clean
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    return image.updateMask(mask).select(BANDS).toInt16().copyProperties(image, ["system:time_start"])

years = [2019, 2020, 2021, 2022, 2023]

city_list = top_10_wgs84['NM_MUN'].tolist()

print("Starting task scheduling")

for city_name in city_list:
    clean_name = city_name.replace(" ", "_").replace("á", "a").replace("í", "i").replace("ó", "o").replace("ã", "a")

    # Exact city geometry
    city_geom = ee_cities.filter(ee.Filter.eq('NM_MUN', city_name)).geometry()

    for year in years:
        # Dry Period
        start_dry = f'{year}-04-01'
        end_dry = f'{year}-09-30'

        dry_coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                    .filterBounds(city_geom)
                    .filterDate(start_dry, end_dry)
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
                    .map(mask_s2_clouds)
                    .median()
                    .clip(city_geom)
                    .unmask(0))

        task_name_dry = f"S2_{clean_name}_{year}_Seco"

        dry_task = ee.batch.Export.image.toDrive(**{
            'image': dry_coll,
            'description': task_name_dry,
            'folder': 'Dataset_Silvicultura',
            'fileNamePrefix': task_name_dry,
            'region': city_geom,
            'scale': 10,
            'crs': 'EPSG:31983',
            'maxPixels': 1e13,
            'formatOptions': {'noData': 0}
        })
        dry_task.start()

        #  Humid Period
        start_humid = f'{year}-10-01'
        end_humid = f'{year+1}-03-31'

        humid_coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                    .filterBounds(city_geom)
                    .filterDate(start_humid, end_humid)
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
                    .map(mask_s2_clouds)
                    .median()
                    .clip(city_geom)
                    .unmask(0))

        task_name_humid = f"S2_{clean_name}_{year}_Umido"

        humid_task = ee.batch.Export.image.toDrive(**{
            'image': humid_coll,
            'description': task_name_humid,
            'folder': 'Dataset_Silvicultura',
            'fileNamePrefix': task_name_humid,
            'region': city_geom,
            'scale': 10,
            'crs': 'EPSG:31983',
            'maxPixels': 1e13,
            'formatOptions': {'noData': 0}
        })
        humid_task.start()

print("Progress: https://code.earthengine.google.com/tasks")

Starting task scheduling
Progress: https://code.earthengine.google.com/tasks
